# Korelasi Lag Curah Hujan Kotak DJF vs Nino-3.4 Rata-Rata Berjalan 3 Bulan

Notebook ini membangun indeks curah hujan DJF dan mengorelasikannya dengan **Nino-3.4 rata-rata berjalan 3 bulan yang terpusat** pada lag -12 sampai +12 untuk:
- **P1**: 1981-2006
- **P2**: 2007-2025

Kurva selisih dihitung sebagai **P2 - P1**.

Kotak curah hujan target:
- lon: **95E to 125E**
- lat: **6S to 2N**

Aturan yang dipakai: jika sudah ada persegi panjang tersimpan yang hampir identik pada definisi domain yang tersedia, notebook akan memakai kotak simpanan tersebut.


## Lag convention (important)

- DJF year uses: **Dec(y-1), Jan(y), Feb(y)** (example: D1980 + JF1981 = DJF1981).
- For each DJF year **y**, **lag 0 is January of year y**, but the ENSO series is a **centered 3-month running mean**.
- Therefore:
  - **lag 0 = DJF(y)** = mean of **Dec(y-1), Jan(y), Feb(y)**
  - **lag -1 = NDJ(y)** = mean of **Nov(y-1), Dec(y-1), Jan(y)**
  - **lag +1 = JFM(y)** = mean of **Jan(y), Feb(y), Mar(y)**
- More generally, lag **k** uses the centered 3-month running-mean Nino-3.4 value at **Jan(y) + k months**.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.stats import pearsonr, norm
from IPython.display import display

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
PATHS_YML = PROJECT_ROOT / 'configs/paths.yml'

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

START_YEAR = 1981
END_YEAR = 2025
P1 = (1981, 2006)
P2 = (2007, 2025)
LAGS = np.arange(-12, 13)

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
})


In [ ]:
def parse_simple_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f'paths.yml not found: {path}')
    out = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        txt = line.strip()
        if not txt or txt.startswith('#'):
            continue
        if ':' not in txt:
            continue
        k, v = txt.split(':', 1)
        out[k.strip()] = v.strip()
    return out


def first_existing_path(candidates):
    for p in candidates:
        if p is not None and p.exists():
            return p
    raise FileNotFoundError('No existing path found in candidates:\n' + '\n'.join(str(p) for p in candidates if p is not None))


def normalize_box(box: dict) -> dict:
    return {
        'lon_min': float(min(box['lon_min'], box['lon_max'])),
        'lon_max': float(max(box['lon_min'], box['lon_max'])),
        'lat_min': float(min(box['lat_min'], box['lat_max'])),
        'lat_max': float(max(box['lat_min'], box['lat_max'])),
    }


def _iter_geo_points(coords):
    if isinstance(coords, (list, tuple)) and len(coords) >= 2 and all(isinstance(v, (int, float)) for v in coords[:2]):
        yield float(coords[0]), float(coords[1])
        return
    if isinstance(coords, (list, tuple)):
        for sub in coords:
            yield from _iter_geo_points(sub)


def extract_boxes_from_geojson(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    features = data.get('features', []) if isinstance(data, dict) else []
    boxes = []
    for i, feat in enumerate(features, start=1):
        points = list(_iter_geo_points((feat.get('geometry') or {}).get('coordinates', [])))
        if not points:
            continue
        xs = np.array([p[0] for p in points], dtype=float)
        ys = np.array([p[1] for p in points], dtype=float)
        props = feat.get('properties') or {}
        name = props.get('name') or props.get('id') or f'feature_{i}'
        boxes.append(
            {
                'name': str(name),
                'lon_min': float(np.nanmin(xs)),
                'lon_max': float(np.nanmax(xs)),
                'lat_min': float(np.nanmin(ys)),
                'lat_max': float(np.nanmax(ys)),
            }
        )
    return boxes


def resolve_box(target_box: dict, domain_json_candidates, edge_tolerance_deg: float = 1.0):
    target = normalize_box(target_box)
    candidates = []
    for p in domain_json_candidates:
        if p.exists():
            try:
                candidates.extend(extract_boxes_from_geojson(p))
            except Exception as e:
                print(f'Warning: failed to parse {p}: {e}')

    best = None
    best_score = np.inf
    for cand in candidates:
        c = normalize_box(cand)
        diffs = [
            abs(c['lon_min'] - target['lon_min']),
            abs(c['lon_max'] - target['lon_max']),
            abs(c['lat_min'] - target['lat_min']),
            abs(c['lat_max'] - target['lat_max']),
        ]
        if max(diffs) <= edge_tolerance_deg:
            score = float(sum(diffs))
            if score < best_score:
                best = cand
                best_score = score

    if best is not None:
        chosen = normalize_box(best)
        return chosen, f"existing stored box: {best.get('name', 'unknown')}"
    return target, 'supervisor fixed box'


def standardize_rainfall_da(da: xr.DataArray) -> xr.DataArray:
    rename_map = {}
    if 'latitude' in da.dims or 'latitude' in da.coords:
        rename_map['latitude'] = 'lat'
    if 'longitude' in da.dims or 'longitude' in da.coords:
        rename_map['longitude'] = 'lon'
    if 'valid_time' in da.dims or 'valid_time' in da.coords:
        rename_map['valid_time'] = 'time'
    if rename_map:
        da = da.rename(rename_map)

    da = da.assign_coords(time=pd.to_datetime(da['time'].values)).sortby('time')
    da = da.assign_coords(lon=(da['lon'] % 360)).sortby('lon')
    return da


def subset_box(da: xr.DataArray, box: dict) -> xr.DataArray:
    box = normalize_box(box)
    lat0, lat1 = float(da['lat'].values[0]), float(da['lat'].values[-1])
    lon0, lon1 = float(da['lon'].values[0]), float(da['lon'].values[-1])

    lat_slice = slice(box['lat_min'], box['lat_max']) if lat0 <= lat1 else slice(box['lat_max'], box['lat_min'])
    lon_slice = slice(box['lon_min'], box['lon_max']) if lon0 <= lon1 else slice(box['lon_max'], box['lon_min'])
    sub = da.sel(lat=lat_slice, lon=lon_slice)
    if sub.sizes.get('lat', 0) == 0 or sub.sizes.get('lon', 0) == 0:
        raise ValueError(f'Empty subset for box={box}')
    return sub


def area_weighted_monthly_mean(da_box: xr.DataArray) -> pd.Series:
    weights = np.cos(np.deg2rad(da_box['lat']))
    box_mean = da_box.weighted(weights).mean(dim=('lat', 'lon'), skipna=True)
    s = box_mean.to_series()
    s.index = pd.to_datetime(s.index)
    s = s.sort_index().groupby(s.index.to_period('M')).mean()
    s.index = s.index.to_timestamp(how='start')
    s.name = 'rain_box_mm_month'
    return s


def load_nino34_monthly(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise ValueError(f'Nino3.4 file must have at least 2 columns, got {df.columns.tolist()}')
    date_col = None
    value_col = None
    for col in df.columns:
        low = str(col).lower()
        if date_col is None and low in {'date', 'time', 'month', 'ym'}:
            date_col = col
        elif value_col is None and low in {'nino34', 'nino3.4', 'nino-3.4', 'anom', 'anomaly', 'value'}:
            value_col = col
    if date_col is None:
        date_col = df.columns[0]
    if value_col is None:
        value_col = df.columns[1]
    s = pd.Series(pd.to_numeric(df[value_col], errors='coerce').to_numpy(), index=pd.to_datetime(df[date_col]))
    s = s.sort_index().groupby(s.index.to_period('M')).mean()
    s.index = s.index.to_timestamp(how='start')
    s.name = 'nino34'
    return s


def build_centered_3mo_running_mean(series: pd.Series) -> pd.Series:
    return series.rolling(3, center=True, min_periods=3).mean().dropna()


def build_djf_rain_index(rain_monthly: pd.Series, start_year: int, end_year: int) -> pd.Series:
    values = []
    index = []
    for year in range(start_year, end_year + 1):
        months = [pd.Timestamp(year - 1, 12, 1), pd.Timestamp(year, 1, 1), pd.Timestamp(year, 2, 1)]
        if all(m in rain_monthly.index for m in months):
            values.append(float(np.nanmean([rain_monthly.loc[m] for m in months])))
            index.append(year)
    return pd.Series(values, index=pd.Index(index, name='djf_year'), name='rain_djf')


def lag_corr_table(djf_rain: pd.Series, nino_3mo: pd.Series, years: np.ndarray, lags: np.ndarray):
    rows = []
    for lag in lags:
        xs = []
        ys = []
        for year in years:
            rkey = int(year)
            nkey = pd.Timestamp(rkey, 1, 1) + pd.DateOffset(months=int(lag))
            if rkey in djf_rain.index and nkey in nino_3mo.index:
                xr_ = djf_rain.loc[rkey]
                yr_ = nino_3mo.loc[nkey]
                if np.isfinite(xr_) and np.isfinite(yr_):
                    xs.append(float(xr_))
                    ys.append(float(yr_))
        if len(xs) >= 3:
            r, p = pearsonr(xs, ys)
        else:
            r, p = np.nan, np.nan
        rows.append({'lag': int(lag), 'r': r, 'p': p, 'n': len(xs)})
    return pd.DataFrame(rows)


def fisher_diff_pvalue(r1, n1, r2, n2):
    if not (np.isfinite(r1) and np.isfinite(r2) and n1 > 3 and n2 > 3):
        return np.nan
    r1 = float(np.clip(r1, -0.999999, 0.999999))
    r2 = float(np.clip(r2, -0.999999, 0.999999))
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    se = np.sqrt(1.0 / (n1 - 3) + 1.0 / (n2 - 3))
    z = (z1 - z2) / se
    return float(2 * norm.sf(abs(z)))


def significance_label(p):
    if not np.isfinite(p):
        return 'n/a'
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return 'ns'


LAG_TITLES = {
    -12: '-DJF',
    -11: '-JFM',
    -10: '-FMA',
    -9: '-MAM',
    -8: '-AMJ',
    -7: '-MJJ',
    -6: '-JJA',
    -5: '-JAS',
    -4: '-ASO',
    -3: '-SON',
    -2: '-OND',
    -1: '-NDJ',
    0: 'DJF',
    1: 'JFM',
    2: 'FMA',
    3: 'MAM',
    4: 'AMJ',
    5: 'MJJ',
    6: 'JJA',
    7: 'JAS',
    8: 'ASO',
    9: 'SON',
    10: 'OND',
    11: 'NDJ',
    12: 'DJF+',
}

MC_MAP_EXTENT = [80, 160, -20, 20]
MC_MAP_XTICKS = np.arange(90, 161, 20)
MC_MAP_YTICKS = np.arange(-20, 21, 10)

def lag_label(lag: int) -> str:
    return LAG_TITLES.get(int(lag), f'{int(lag):+d}')

def style_lag_axis(ax, ylim=(-1.0, 1.0)):
    ax.set_xlim(-12.5, 12.5)
    ax.set_ylim(*ylim)
    ax.set_xticks(LAGS)
    ax.set_xticklabels([lag_label(l) for l in LAGS], rotation=45, ha='right')
    ax.set_yticks(np.arange(ylim[0], ylim[1] + 1e-9, 0.2))
    ax.tick_params(axis='both', which='both', direction='out', top=True, right=True, labelsize=11, length=4, width=0.8)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.35)

def style_mc_map(ax):
    ax.set_extent(MC_MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='#eaf2f8', zorder=0)
    ax.add_feature(cfeature.LAND, facecolor='#f7f4ee', edgecolor='none', zorder=1)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.45, color='black', zorder=3)
    ax.gridlines(draw_labels=False, linewidth=0.35, color='gray', alpha=0.45, linestyle='--')
    ax.set_xticks(MC_MAP_XTICKS, crs=ccrs.PlateCarree())
    ax.set_yticks(MC_MAP_YTICKS, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')


In [ ]:
# 1) Resolve paths from configs/paths.yml
cfg = parse_simple_yaml(PATHS_YML)
climate_data_dir = Path(cfg.get('climate_data_dir', '')) if cfg.get('climate_data_dir') else None
project_data_dir = Path(cfg.get('project_data_dir', '')) if cfg.get('project_data_dir') else None

rain_candidates = [
    (climate_data_dir / 'mswep-monthly/mswep_monthly_combined.nc') if climate_data_dir else None,
    (project_data_dir / 'mswep-monthly/mswep_monthly_combined.nc') if project_data_dir else None,
    PROJECT_ROOT / 'external/ClimateData/mswep-monthly/mswep_monthly_combined.nc',
]

nino_candidates = [
    (climate_data_dir / 'index-monthly/nino34.anom.csv') if climate_data_dir else None,
    (project_data_dir / 'index-monthly/nino34.anom.csv') if project_data_dir else None,
    PROJECT_ROOT / 'external/ClimateData/index-monthly/nino34.anom.csv',
]

domain_json_candidates = [
    (project_data_dir / 'all_data/domain.json') if project_data_dir else None,
    PROJECT_ROOT / 'data/all_data/domain.json',
]
domain_json_candidates = [p for p in domain_json_candidates if p is not None]

RAIN_PATH = first_existing_path(rain_candidates)
NINO34_PATH = first_existing_path(nino_candidates)
BOX, BOX_SOURCE = resolve_box(TARGET_BOX, domain_json_candidates, edge_tolerance_deg=1.0)

# 2) Load monthly MSWEP rainfall and monthly Nino-3.4, then build centered 3-month running mean
ds_rain = xr.open_dataset(RAIN_PATH)
rain_var = 'precipitation' if 'precipitation' in ds_rain.data_vars else list(ds_rain.data_vars)[0]
rain = standardize_rainfall_da(ds_rain[rain_var])
nino_monthly = load_nino34_monthly(NINO34_PATH)
nino_3mo = build_centered_3mo_running_mean(nino_monthly)

# Keep the monthly span needed by DJF 1981..2025 and lags -12..+12 around Jan DJF year.
rain = rain.sel(time=slice('1979-12-01', '2025-02-28'))
nino_3mo = nino_3mo.loc['1980-01-01':'2026-01-01']

# 3) Subset rainfall to box and compute area-weighted monthly box mean
rain_box = subset_box(rain, BOX)
rain_box_monthly = area_weighted_monthly_mean(rain_box)

# 4) Build DJF rainfall index
djf_rain = build_djf_rain_index(rain_box_monthly, START_YEAR, END_YEAR)

print('Rainfall path:', RAIN_PATH)
print('Nino3.4 path:', NINO34_PATH)
print('Box source:', BOX_SOURCE)
print('Box used:', BOX)
print('ENSO series used: centered 3-month running-mean Nino-3.4')


In [ ]:
# 5) Lag-correlation curves for P1 and P2
p1_years = np.arange(P1[0], P1[1] + 1)
p2_years = np.arange(P2[0], P2[1] + 1)

tbl_p1 = lag_corr_table(djf_rain, nino_3mo, p1_years, LAGS).rename(columns={'r': 'r_p1', 'p': 'p_p1', 'n': 'n_p1'})
tbl_p2 = lag_corr_table(djf_rain, nino_3mo, p2_years, LAGS).rename(columns={'r': 'r_p2', 'p': 'p_p2', 'n': 'n_p2'})

result = tbl_p1.merge(tbl_p2, on='lag', how='inner')
result['difference_p2_minus_p1'] = result['r_p2'] - result['r_p1']
result['p_diff'] = [
    fisher_diff_pvalue(r1, n1, r2, n2)
    for r1, n1, r2, n2 in zip(result['r_p1'], result['n_p1'], result['r_p2'], result['n_p2'])
]

result['sig_p1'] = result['p_p1'].map(significance_label)
result['sig_p2'] = result['p_p2'].map(significance_label)
result['sig_diff'] = result['p_diff'].map(significance_label)

table_cols = [
    'lag',
    'r_p1', 'r_p2', 'difference_p2_minus_p1',
    'p_p1', 'p_p2', 'p_diff',
    'sig_p1', 'sig_p2', 'sig_diff',
    'n_p1', 'n_p2',
]

table_out = result[table_cols].copy()
table_out[['r_p1', 'r_p2', 'difference_p2_minus_p1', 'p_p1', 'p_p2', 'p_diff']] = (
    table_out[['r_p1', 'r_p2', 'difference_p2_minus_p1', 'p_p1', 'p_p2', 'p_diff']].round(4)
)

print('Tabel lag: lag, r P1, r P2, selisih (P2-P1), dan p-value')
display(table_out)


## Peta Kotak Curah Hujan

Kotak di bawah adalah wilayah curah hujan yang dipakai untuk perhitungan korelasi lag.


In [ ]:
# 6) Rainfall box map used in the correlation computation
fig = plt.figure(figsize=(9.6, 4.8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))
style_mc_map(ax)
box_lons = [TARGET_BOX['lon_min'], TARGET_BOX['lon_max'], TARGET_BOX['lon_max'], TARGET_BOX['lon_min'], TARGET_BOX['lon_min']]
box_lats = [TARGET_BOX['lat_min'], TARGET_BOX['lat_min'], TARGET_BOX['lat_max'], TARGET_BOX['lat_max'], TARGET_BOX['lat_min']]
ax.plot(box_lons, box_lats, color='tab:red', linewidth=2.5, transform=ccrs.PlateCarree(), zorder=5)
ax.fill(box_lons, box_lats, color='tab:red', alpha=0.12, transform=ccrs.PlateCarree(), zorder=4)
ax.text(TARGET_BOX['lon_min'] + 0.8, TARGET_BOX['lat_max'] + 0.7, 'Rainfall box', color='tab:red', fontsize=12, fontweight='bold', transform=ccrs.PlateCarree(), zorder=6)
ax.set_title('Kotak curah hujan yang dipakai untuk korelasi lag DJF', fontsize=14, fontweight='bold', loc='left', pad=20)
plt.tight_layout()
plt.show()


In [ ]:
# 7) Kurva P1 dan P2
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(result['lag'], result['r_p1'], marker='o', markersize=4.5, linewidth=2.0, color='tab:blue', label='P1 (1981-2006)')
ax.plot(result['lag'], result['r_p2'], marker='o', markersize=4.5, linewidth=2.0, color='tab:orange', label='P2 (2007-2025)')
ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--', alpha=0.8)
ax.axvline(0.0, color='gray', linewidth=1.0, linestyle=':', alpha=0.8)
ax.set_xlabel('Lag relatif terhadap kotak curah hujan DJF')
ax.set_ylabel('Koefisien korelasi (r)')
ax.set_title('Korelasi lag: indeks kotak curah hujan DJF vs Nino-3.4 rata-rata berjalan 3 bulan', pad=20)
style_lag_axis(ax)
ax.legend(frameon=False, ncols=2, loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# 8) Kurva Selisih (P2 - P1)
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(result['lag'], result['difference_p2_minus_p1'], marker='o', markersize=4.5, linewidth=2.0, color='tab:purple', label='Selisih (P2 - P1)')

sig_mask = result['p_diff'] < 0.05
ax.scatter(
    result.loc[sig_mask, 'lag'],
    result.loc[sig_mask, 'difference_p2_minus_p1'],
    color='crimson',
    s=50,
    zorder=3,
    label='p_diff < 0.05'
)

ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--', alpha=0.8)
ax.axvline(0.0, color='gray', linewidth=1.0, linestyle=':', alpha=0.8)
ax.set_xlabel('Lag relatif terhadap kotak curah hujan DJF')
ax.set_ylabel('Selisih koefisien korelasi (P2 - P1)')
ax.set_title('Selisih kurva korelasi lag: P2 - P1', pad=20)
style_lag_axis(ax, ylim=(-1.0, 1.4))
ax.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()
